# Hugging Face to Spark NLP: ONNX Conversion Guide

This notebook walks you through converting any compatible Hugging Face model to a Spark NLP compatible format using ONNX.
What this covers:
- Exporting a Hugging Face model to ONNX using optimum
- Preparing model assets (vocab, labels, tokenizer files)
- Loading the exported model into Spark NLP
- Running an end-to-end inference pipeline

# onnx setup/conversion

In [ ]:
!pip install --upgrade -q "optimum-onnx[onnxruntime]" optimum[onnx]

Name of the model you want to convert from Hugging Face: https://huggingface.co/models

In [ ]:
model_id = "google-bert/bert-base-uncased"

In [ ]:
!optimum-cli export onnx \
  --model $model_id onnx \
  --trust-remote-code

run for more info: `!optimum-cli export onnx --help`

In [ ]:
# @title assets prep function
# This function prepares the assets for an ONNX exported model.

# It does the following:
# - Creates an `assets` folder inside the model directory.
# - Moves all files except `.onnx` files into `assets`.
# - Generates `vocab.txt` from `vocab.json` or `tokenizer.json` if missing.
# - Generates `labels.txt` from `config.json` if the model is for classification.
# - If no tokenizer files exist, it can download them from HuggingFace using the provided model ID.

import os, json, re, shutil, time, requests
from pathlib import Path

from transformers import AutoTokenizer
from google.colab import files, userdata

def prepare_assets(
    model_path: str = "onnx",
    classification: bool = False,
    which_tokenizer: str = None
):
    model_path = Path(model_path)
    assets_path = model_path / "assets"
    os.makedirs(assets_path, exist_ok=True)

    has_assets = any(model_path.glob("*.json")) or any(model_path.glob("*.txt"))
    if not has_assets and which_tokenizer:
        print(f"No assets found in {model_path}, downloading tokenizer '{which_tokenizer}'...")
        tokenizer = AutoTokenizer.from_pretrained(which_tokenizer, use_fast=False)
        tokenizer.save_pretrained(assets_path)

    for file_path in model_path.iterdir():
        if file_path.is_file() and file_path.suffix != ".onnx":
            file_path.rename(assets_path / file_path.name)

    vocab_txt = assets_path / "vocab.txt"
    if not vocab_txt.exists():
        vocab_json = assets_path / "vocab.json"
        tokenizer_json = assets_path / "tokenizer.json"

        if vocab_json.exists():
            with open(vocab_json, "r", encoding="utf-8") as f, open(vocab_txt, "w", encoding="utf-8") as out:
                vocab = json.load(f)
                for token in vocab.keys():
                    out.write(token + "\n")
            print("Generated vocab.txt from vocab.json")
        elif tokenizer_json.exists():
            with open(tokenizer_json, "r", encoding="utf-8") as f, open(vocab_txt, "w", encoding="utf-8") as out:
                vocab = json.load(f).get("model", {}).get("vocab", [])

                for e in vocab:
                    token = (
                        e if isinstance(e, str)
                        else e[0] if isinstance(e, (list, tuple)) and e
                        else e.get("token") if isinstance(e, dict)
                        else None
                    )
                    if token is not None:
                        out.write(token + "\n")
            print("Generated vocab.txt from tokenizer.json")

    if classification:
        config_json = assets_path / "config.json"
        labels_txt = assets_path / "labels.txt"
        if config_json.exists():
            with open(config_json, "r", encoding="utf-8") as f:
                labels = json.load(f).get("id2label", {}).values()
            with open(labels_txt, "w", encoding="utf-8") as f:
                f.write("\n".join(labels))
            print("Generated labels.txt from config.json")

    print(f"Assets preparation complete at {assets_path}")


In [ ]:
!ls -lh onnx

In [ ]:
# Note: If no tokenizer files were downloaded ^, you may need to explicitly set `which_tokenizer` to the original base model ID.
# For example, when converting a fine-tuned model like `roberta-base-finetuned-news`,
# the tokenizer is often inherited from the base model and not included with the fine-tuned checkpoint.
# In this case, set `which_tokenizer="roberta-base"` to ensure the correct tokenizer files are retrieved.
# This is usually only required for niche or custom fine-tuned models. Most standard models include their tokenizer files by default.


In [ ]:
prepare_assets(
    classification=True,
    which_tokenizer=model_id
)


In [ ]:
!ls -lh onnx onnx/assets

# import in sparknlp

In [ ]:
# This is only to setup PySpark and Spark NLP on Colab
!wget -q http://setup.johnsnowlabs.com/colab.sh -O - | bash

In [ ]:
import sparknlp

spark = sparknlp.start()

print("Spark NLP version: ", sparknlp.version())
print("Apache Spark version: ", spark.version)

specify the name of the model you want in sparknlp here

In [ ]:
sparknlp_model_name = "bert_base_uncased"

Dont forget `.setStorageRef()` here if its an embedding - it should be the same as the model name so

`.setStorageRef(sparknlp_model_name)`

In [ ]:
from sparknlp.annotator import BertEmbeddings

imported_model = (
    BertEmbeddings.loadSavedModel("onnx", spark)
      .setInputCols(["token", "document"])
      .setOutputCol("bert_embeddings")
      .setStorageRef(sparknlp_model_name)
)


In [ ]:
print(imported_model.explainParams())

In [ ]:
imported_model.write().overwrite().save(sparknlp_model_name)

In [ ]:
!ls -lh $sparknlp_model_name

test imported model

In [ ]:
from sparknlp.base import *
from sparknlp.annotator import *
from pyspark.ml import Pipeline

documentAssembler = DocumentAssembler() \
    .setInputCol("text") \
    .setOutputCol("document")

tokenizer = Tokenizer() \
    .setInputCols("document") \
    .setOutputCol("token")

loaded_model = BertEmbeddings \
    .load(sparknlp_model_name) \
    .setInputCols(["token", "document"]) \
    .setOutputCol("bert_embeddings")

pipeline = Pipeline(stages=[
    documentAssembler,
    tokenizer,
    loaded_model
])

data = spark.createDataFrame([["Covid cases are increasing fast!"]]).toDF("text")

result = pipeline.fit(data).transform(data)
result.select("bert_embeddings.embeddings").show()
